# CPT_S 475 Semester Project
## Team: Project Hail Mary

**Team Members:**

*- Finley Blaylock*

*- Noah Manuel*

**Project Goal:**
Create three different machine-learning (ML) programs trained on the Kaggle [Kepler exoplanet dataset](https://www.kaggle.com/datasets/gauravkumar2525/kepler-exoplanet-dataset/data). 

Targeted ML programs:
- Decision Tree
- Random Forest Classifier
- Logistical Regression

## Part 0: Pre-requisites and Project Materials

In [1]:
# Install once from a terminal if needed: python -m pip install -r requirements.txt

In [2]:
""" GLOBAL IMPORTS """

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl

## Part 1: Data Cleansing

**Section Goal:** 
Take the dataset (located in `./data/exoplanets_data.csv`) and cleanse the data for any missing values and errors that can affect the models training and testing data. 

**Cleaning assumptions:** The following is the partner-authored cleaning procedure. Matching a column mean suggests prior imputation but does not prove it; thresholds based on the complete CSV also make this a preliminary, dataset-dependent analysis. The period/insolation exclusions are quality-control choices, not proof that an object cannot be a planet. Large radii do not by themselves establish an eclipsing binary. The supplied numeric class mapping is a team assumption pending source verification.

In [3]:
DATA_PATH = "./data/exoplanets_data.csv"
CLEANSED_DATA_PATH = "./data/cleansed_exoplanets_data.csv"


DROP_KOI_SCORE = True


rawData = pd.read_csv(DATA_PATH)
rawDataFrame = rawData.copy()

print("Raw shape: ", rawDataFrame.shape)
print("NaN cells:", int(rawDataFrame.isna().sum().sum()))
print("Duplicate Rows", rawDataFrame.duplicated().sum())
print("Target counts (0=False Positive, 1=Candidate, 2=Confirmed): ", rawDataFrame['koi_disposition'].value_counts().sort_index().to_string())

Raw shape:  (9564, 12)
NaN cells: 0
Duplicate Rows 0
Target counts (0=False Positive, 1=Candidate, 2=Confirmed):  koi_disposition
0    4839
1    1982
2    2743


In [4]:
# Restorinng the original data frame to the raw data frame
"""
There are no NaN values in the dataset. However, there is a chance that these missing values were filled with the mean
of the column as a placeholder. Therefore, we have to restore these placedholder values with NaN to ensure that the training data isn't misrepresented. The following code will restore the NaN values in the dataset to ensure that the  data is 
accurate and representative of the actual data.
"""

FEATURES = ["koi_score", "koi_prad", "koi_teq", "koi_insol", "koi_steff", "koi_srad", "koi_slogg"]
CORE = ["koi_prad", "koi_teq", "koi_steff", "koi_srad", "koi_slogg"]

for c in FEATURES:
    # 1. Calculate the mean of the column
    col_mean = rawDataFrame[c].mean()
    
    # 2. Find where values are close to the mean (returns a boolean NumPy array)
    is_placeholder = np.isclose(rawDataFrame[c], col_mean, rtol=0, atol=1e-9)
    
    # 3. Use .values to guarantee a clean array for .loc and avoid Series hashing issues
    rawDataFrame.loc[is_placeholder, c] = np.nan

print("Number of NaN cells after restoring placeholders: ", int(rawDataFrame.isna().sum().sum()))

print("Shape of the DataFrame after reintroducing NaNs: ", rawDataFrame.shape)

Number of NaN cells after restoring placeholders:  3646
Shape of the DataFrame after reintroducing NaNs:  (9564, 12)


In [5]:
# Dropping unusable rows 

# Rows where every core planet/star measurement was a placeholder carry no information.
# They are 99% False Positive/Candidate, so keeping them would teach the models that "average values" = "not a planet".
no_measurements = rawDataFrame[CORE].isna().all(axis=1)
print("Rows with no core measurements:", no_measurements.sum())
print(rawDataFrame.loc[no_measurements, "koi_disposition"].value_counts().sort_index().to_string())
rawDataFrame = rawDataFrame.loc[~no_measurements]

# Physically invalid rows: Kepler observed for ~4 years, so a ~130,000-day period with zero
# insolation is a catalog error, not a planet.
invalid = (rawDataFrame["koi_insol"] <= 0) | (rawDataFrame["koi_period"] > 10000)
print("\nPhysically invalid rows:", invalid.sum(), "->", rawDataFrame.loc[invalid, "kepoi_name"].tolist())
rawDataFrame = rawDataFrame.loc[~invalid]

# extreme radii (100s-1000s of Earth radii) are kept. They are real eclipsing binaries
# labeled False Positive and carry signal; handle their skew with scaling/log transforms later.

print("Shape of dataframe after dropping unusable rows: ")
print(rawDataFrame.shape)

Rows with no core measurements: 363
koi_disposition
0    257
1    104
2      2

Physically invalid rows: 1 -> ['K01174.01']
Shape of dataframe after dropping unusable rows: 
(9200, 12)


In [6]:
# Final Dataframe shaping

if DROP_KOI_SCORE:
    rawDataFrame = rawDataFrame.drop(columns="koi_score")

# Identifiers are not features, but kepid is kept so Part 2 can split by star (several KOIs share one star)
rawDataFrame = rawDataFrame.reset_index(drop=True)

# Sanity checks
assert rawDataFrame.isna().sum().sum() == 0 or not DROP_KOI_SCORE, "unexpected NaNs remain"
assert rawDataFrame["kepoi_name"].is_unique
assert rawDataFrame["koi_disposition"].isin([0, 1, 2]).all()
assert (rawDataFrame[["koi_period", "koi_prad", "koi_teq", "koi_insol", "koi_steff", "koi_srad"]] > 0).all().all()

rawDataFrame.to_csv(CLEANSED_DATA_PATH, index=False)

summary = pd.DataFrame({"raw": rawData["koi_disposition"].value_counts().sort_index(),
                        "clean": rawDataFrame["koi_disposition"].value_counts().sort_index()})
summary["clean_%"] = (summary["clean"] / summary["clean"].sum() * 100).round(1)
print(f"Rows: {len(rawData)} -> {len(rawDataFrame)} ({len(rawData) - len(rawDataFrame)} removed) | columns: {rawDataFrame.shape[1]}")
print(f"Remaining NaN cells: {int(rawDataFrame.isna().sum().sum())}")
print(f"Saved to {CLEANSED_DATA_PATH}\n")
print(summary.to_string())

Rows: 9564 -> 9200 (364 removed) | columns: 11
Remaining NaN cells: 0
Saved to ./data/cleansed_exoplanets_data.csv

                  raw  clean  clean_%
koi_disposition                      
0                4839   4582     49.8
1                1982   1877     20.4
2                2743   2741     29.8


## Part 2: Training/Evaluation Data Creation

**Section Goal:** Split the dataset with a 70% training and 30% evaluation configuration. Each model will be trained on the same train-test dataset.

**Classifier Goal:** Predict `koi_disposition` (0 = False Positive, 1 = Candidate, 2 = Confirmed) from cleaned planet and star features. All models will use the same cleansed 70/30 train-test dataset so results are directly comprable. 

In [7]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
from pathlib import Path
import json
from analysis import features

cleansedDataFrame = pd.read_csv(CLEANSED_DATA_PATH)

In [8]:
# Keep all candidates from each star on the same side of the split.
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=475).split(
    cleansedDataFrame, groups=cleansedDataFrame.kepid))
assert not set(cleansedDataFrame.iloc[train_idx].kepid) & set(cleansedDataFrame.iloc[test_idx].kepid)
keplerFeatures = features(cleansedDataFrame, engineered=True)
keplerFeatures_train = keplerFeatures.iloc[train_idx]
keplerFeatures_test = keplerFeatures.iloc[test_idx]
keplerLabel_train = cleansedDataFrame.koi_disposition.iloc[train_idx]
keplerLabel_test = cleansedDataFrame.koi_disposition.iloc[test_idx]
training_groups = cleansedDataFrame.kepid.iloc[train_idx]
assert set(keplerLabel_train) == set(keplerLabel_test) == {0, 1, 2}
assert np.isfinite(keplerFeatures.to_numpy()).all()
assert not set(keplerFeatures) & {'kepid', 'kepoi_name', 'koi_score', 'koi_disposition'}
print('Training features:', keplerFeatures_train.shape)
print('Evaluation features:', keplerFeatures_test.shape)
display(pd.crosstab(cleansedDataFrame.koi_disposition, np.where(np.isin(np.arange(len(cleansedDataFrame)), test_idx), 'evaluation', 'training')))

OUTPUT = Path('results_cleaned')
OUTPUT.mkdir(exist_ok=True)
comparison = []
model_reports = {}
fitted_models = {}
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=475)

def evaluate_model(name, estimator):
    # Imputation/scaling fit independently inside each training fold.
    model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), estimator)
    scores = cross_val_score(model, keplerFeatures_train, keplerLabel_train,
                             groups=training_groups, cv=cv, scoring='f1_macro')
    model.fit(keplerFeatures_train, keplerLabel_train)
    pred = model.predict(keplerFeatures_test)
    comparison.append({'model': name, 'cv_macro_f1': scores.mean(), 'cv_fold_sd': scores.std(),
                       'training_macro_f1': f1_score(keplerLabel_train, model.predict(keplerFeatures_train), average='macro'),
                       'accuracy': accuracy_score(keplerLabel_test, pred),
                       'macro_f1': f1_score(keplerLabel_test, pred, average='macro')})
    report = classification_report(keplerLabel_test, pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(keplerLabel_test, pred, labels=[0,1,2])
    report['confusion_matrix'] = cm.tolist()
    report['one_vs_rest'] = {str(i): {'TP': int(cm[i,i]), 'FP': int(cm[:,i].sum()-cm[i,i]),
        'FN': int(cm[i,:].sum()-cm[i,i]), 'TN': int(cm.sum()-cm[:,i].sum()-cm[i,:].sum()+cm[i,i])} for i in range(3)}
    model_reports[name] = report
    fitted_models[name] = model
    print(name)
    print(classification_report(keplerLabel_test, pred, zero_division=0))
    display(pd.DataFrame(report['one_vs_rest']).T)
    return model

baseline = evaluate_model('Majority baseline', DummyClassifier(strategy='most_frequent'))

Training features: (6436, 10)
Evaluation features: (2764, 10)


col_0,evaluation,training
koi_disposition,,
0,1373,3209
1,562,1315
2,829,1912


Majority baseline
              precision    recall  f1-score   support

           0       0.50      1.00      0.66      1373
           1       0.00      0.00      0.00       562
           2       0.00      0.00      0.00       829

    accuracy                           0.50      2764
   macro avg       0.17      0.33      0.22      2764
weighted avg       0.25      0.50      0.33      2764



,TP,FP,FN,TN
0,1373,1391,0,0
1,0,0,562,2202
2,0,0,829,1935


**Features:** Eight catalog predictors become ten inputs: log1p transformations of period, planet radius, equilibrium temperature, insolation, and stellar radius; plus log1p planet/star radius ratio and relative luminosity proxy. See `analysis.features` for unit conversion. `koi_score` and identifiers are excluded. The CSV contains catalog measurements, not raw light curves. Grouping yields approximately 70/30 rows, rather than an exact row percentage.

## Part 3: Decision Tree Classifier

**Section Goal:** Train a Decision Tree Classifier on the train-test dataset to evaluate performance.

**Why use Decision Tree:** A tree captures nonlinear thresholds and feature interactions. Depth and leaf-size limits reduce overfitting.

In [9]:
decision_tree = evaluate_model('Decision Tree', DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, class_weight='balanced', random_state=475))

Decision Tree
              precision    recall  f1-score   support

           0       0.89      0.63      0.74      1373
           1       0.40      0.60      0.48       562
           2       0.67      0.77      0.72       829

    accuracy                           0.66      2764
   macro avg       0.65      0.66      0.64      2764
weighted avg       0.72      0.66      0.68      2764



,TP,FP,FN,TN
0,867,112,506,1279
1,335,500,227,1702
2,636,314,193,1621


## Part 4: Random Forest Classifier

**Section Goal:** Train a Random Forest Classifier on the train-test dataset to evaluate performance.

**Why use Random Forest:** Averaging many trees can reduce variance and capture nonlinear relationships.

In [10]:
random_forest = evaluate_model('Random Forest', RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced', random_state=475, n_jobs=-1))

Random Forest
              precision    recall  f1-score   support

           0       0.86      0.72      0.78      1373
           1       0.45      0.56      0.50       562
           2       0.71      0.77      0.74       829

    accuracy                           0.70      2764
   macro avg       0.67      0.68      0.67      2764
weighted avg       0.73      0.70      0.71      2764



,TP,FP,FN,TN
0,992,165,381,1226
1,314,388,248,1814
2,640,265,189,1670


## Part 5: Logistic Regression

**Section Goal:** Train a Logisitc Regression Classifier on the train-test dataset to evaluate performance.

**Why use Logistic Regression:** A regularized linear classifier provides a simpler comparison. Log transforms and scaling address uneven feature magnitudes.

In [11]:
logistic_regression = evaluate_model('Logistic Regression', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=475))

Logistic Regression
              precision    recall  f1-score   support

           0       0.88      0.63      0.73      1373
           1       0.41      0.54      0.47       562
           2       0.62      0.78      0.69       829

    accuracy                           0.66      2764
   macro avg       0.64      0.65      0.63      2764
weighted avg       0.71      0.66      0.67      2764



,TP,FP,FN,TN
0,859,116,514,1275
1,305,441,257,1761
2,648,395,181,1540


## Part 6: Model Performance Comparison

**Section Goals:** 
- Compare the accuracy, precision, and false-positive (FP), false-negative (FN), true-negative (TN), true-positive (TP) rates of each model.
- Compare the strengths and weakness of each model 

In [12]:
metrics = pd.DataFrame(comparison)
display(metrics.round(4))
metrics.to_csv(OUTPUT / 'metrics.csv', index=False)
(OUTPUT / 'classification_reports.json').write_text(json.dumps(model_reports, indent=2))
pd.DataFrame({'kepoi_name': cleansedDataFrame.kepoi_name, 'kepid': cleansedDataFrame.kepid,
              'split': np.where(np.isin(np.arange(len(cleansedDataFrame)), test_idx), 'evaluation', 'training')}).to_csv(OUTPUT / 'split.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, name in zip(axes, ['Decision Tree', 'Random Forest', 'Logistic Regression']):
    ConfusionMatrixDisplay(np.array(model_reports[name]['confusion_matrix'])).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
fig.tight_layout(); fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
counts = cleansedDataFrame.koi_disposition.value_counts().sort_index()
bars = ax.bar([f'Class {i}' for i in counts.index], counts.values)
ax.bar_label(bars, padding=3); ax.set(ylabel='Records', title='Class distribution after partner cleaning')
fig.tight_layout(); fig.savefig(OUTPUT / 'class_distribution.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8,5))
for label in [0,1,2]:
    subset = cleansedDataFrame.iloc[train_idx].query('koi_disposition == @label')
    ax.scatter(subset.koi_period, subset.koi_prad, s=9, alpha=.35, label=f'Class {label}')
ax.set(xscale='log', yscale='log', xlabel='Period (days)', ylabel='Radius (Earth radii)', title='Training catalog measurements')
ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT / 'period_radius.png', dpi=150); plt.show()


,model,cv_macro_f1,cv_fold_sd,training_macro_f1,accuracy,macro_f1
0,Majority baseline,0.2218,0.0000,0.2218,0.4967,0.2213
1,Decision Tree,0.6286,0.0170,0.7025,0.6650,0.6440
2,Random Forest,0.6727,0.0138,0.9128,0.7041,0.6731
3,Logistic Regression,0.6343,0.0270,0.6386,0.6556,0.6301


C:\Users\hazop\AppData\Local\Temp\ipykernel_26232\3474997741.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=150); plt.show()
C:\Users\hazop\AppData\Local\Temp\ipykernel_26232\3474997741.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUT / 'class_distribution.png', dpi=150); plt.show()


C:\Users\hazop\AppData\Local\Temp\ipykernel_26232\3474997741.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT / 'period_radius.png', dpi=150); plt.show()


**Interpretation:** Class imbalance motivates macro-F1 alongside accuracy. Period and radius span orders of magnitude and show overlapping classes. Compare training and evaluation F1 for overfitting; small differences on one split are not conclusive. CV uses training stars only, but the inherited mean-placeholder rule was derived from the full CSV and requires provenance verification. These cleaned-data results are separate from the earlier full-data experiment in `results/`; they use a different sample and must not be treated as a controlled feature ablation.